# Intraday Backtest: Momentum + ATR + RSI (5m)

This notebook fetches 5-minute bars, runs the combined scanner (momentum z-score + ATR breakout + RSI flip),
and runs a simple vectorbt backtest to report PnL, Sharpe, drawdown, average signals/day, and per-regime performance.

In [ ]:
# Install requirements if needed (run in notebook environment)
# !pip install -r ../requirements.txt
# vectorbt may require extra dependencies; install if absent.
import pandas as pd
import numpy as np
import vectorbt as vbt
from src.data.fetchers import fetch_prices
from src.engine.combined_scanner import scan_with_extra_signals


In [ ]:
# Fetch 5-minute data for a small universe
tickers = ['SPY','QQQ','IWM']
prices = fetch_prices(tickers, interval='5m', period='14d')
# use same prices as high/low approximation for demo purposes
high = prices.copy()
low = prices.copy()
prices = prices.ffill().dropna(how='all')
print('Data shape:', prices.shape)


In [ ]:
# Run combined scanner (aim ~10 signals/day, allow down to 0.6 confidence)
signals_dict = scan_with_extra_signals(prices, high=high, low=low, window_short=3, window_long=12, zscore_window=200, threshold=1.0, target_min_signals_per_day=10, min_threshold=0.6, atr_window=14, atr_lookback=20, atr_mult=1.0, rsi_window=8, weights={'momentum':1.0,'atr':1.0,'rsi':1.0}, vote_threshold=0.4)
combined = signals_dict['combined']
print('Average signals/day (combined):', combined.abs().sum(axis=1).groupby(combined.index.date).sum().mean())


In [ ]:
# Build entries/exits for vectorbt (long/short)
# Long entry when combined == 1, exit when combined <= 0
entries = combined == 1
exits = combined <= 0
short_entries = combined == -1
short_exits = combined >= 0
# Ensure alignment
close = prices
# Run portfolio simulation with simple fees/slippage assumptions
pf = vbt.Portfolio.from_signals(close, entries, exits, short_entries=short_entries, short_exits=short_exits, freq='5T', fees=0.0005, slippage=0.0002, init_cash=100000)
# Performance summary
print(pf.total_return())
print('Sharpe (annualized):', pf.sharpe_ratio())
print('Max drawdown:', pf.max_drawdown())
print('Average signals/day:', combined.abs().sum(axis=1).groupby(combined.index.date).sum().mean())


In [ ]:
# Per-regime performance: if you have macro regime series aligned to prices, you can split performance.
# For demo we synthesize a simple alternating regime based on hour-of-day (not realistic). Replace with FRED-derived regime for real tests.
regime = (combined.index.hour >= 9) & (combined.index.hour <= 16)
regime = pd.Series(regime.astype(int), index=combined.index)
# compute daily PnL series from portfolio
daily_nav = pf.daily_nav()
daily_returns = daily_nav.pct_change().dropna()
# crude grouping by regime on the day of signals (demo only)
print('Daily avg return (demo regime=1):', daily_returns[regime.resample('D').max().astype(bool)].mean())
